# Sistema CRM Bancario: Reglas SI-ENTONCES

**Curso**: Sistemas basados en conocimiento  
**Profesor**: Alexander Barrantes Calderón  
**Estudiante**: Erick Edgardo Salas Chaverri

## Objetivo
Este notebook implementa el motor de inferencia del sistema CRM bancario con **45 reglas de decisión** basadas en lógica SI-ENTONCES (IF-THEN) y frames.

El sistema automatiza decisiones personalizadas de interacción con clientes combinando cinco dimensiones: **RFM, Arquetipos, Customer Journey, Afinidad de Productos y Canales de Contacto**.

In [1]:
import json
import os

# reglas en el formato de SI-ENTONCES - 45 REGLAS COMPLETAS
REGLAS_JSON = '''[
  {
    "id": "R1",
    "si": [
      "rfm_monetary_alto",
      "arquetipo_cliente_patrimonial",
      "customer_journey_crecimiento",
      "afinidad_fondos_inversion",
      "canal_asesor_personal"
    ],
    "entonces": "Ofrecer productos de inversión premium"
  },
  {
    "id": "R2",
    "si": [
      "rfm_monetary_bajo",
      "arquetipo_profesional_joven_digital",
      "customer_journey_activacion",
      "afinidad_tarjeta_credito",
      "canal_aplicacion_movil"
    ],
    "entonces": "Incentivos para activación de tarjeta digital"
  },
  {
    "id": "R3",
    "si": [
      "rfm_monetary_medio",
      "arquetipo_familia_en_expansion",
      "customer_journey_madurez",
      "afinidad_seguros",
      "canal_email"
    ],
    "entonces": "Oferta integral de seguros familiares"
  },
  {
    "id": "R4",
    "si": [
      "rfm_monetary_bajo",
      "arquetipo_cliente_transaccional",
      "customer_journey_reactivacion",
      "afinidad_cuenta_corriente",
      "canal_sms"
    ],
    "entonces": "SMS de reactivación con promoción básica"
  },
  {
    "id": "R5",
    "si": [
      "rfm_monetary_alto",
      "arquetipo_cliente_patrimonial",
      "customer_journey_riesgo_de_abandono",
      "afinidad_fondos_inversion",
      "canal_asesor_personal"
    ],
    "entonces": "Retención estratégica con gestor dedicado"
  },
  {
    "id": "R6",
    "si": [
      "rfm_monetary_medio",
      "arquetipo_emprendedor_pequeño_empresario",
      "customer_journey_crecimiento",
      "afinidad_credito_personal",
      "canal_telefono"
    ],
    "entonces": "Consultoría de crédito comercial personalizada"
  },
  {
    "id": "R7",
    "si": [
      "rfm_monetary_bajo",
      "arquetipo_familia_en_expansion",
      "customer_journey_adquisicion",
      "afinidad_cuenta_corriente",
      "canal_sucursal"
    ],
    "entonces": "Bienvenida familiar con cuentas básicas"
  },
  {
    "id": "R8",
    "si": [
      "rfm_monetary_alto",
      "arquetipo_profesional_joven_digital",
      "customer_journey_madurez",
      "afinidad_fondos_inversion",
      "canal_aplicacion_movil"
    ],
    "entonces": "Plataforma avanzada de inversiones digital"
  },
  {
    "id": "R9",
    "si": [
      "rfm_monetary_alto",
      "frequency_alta",
      "arquetipo_cliente_patrimonial",
      "customer_journey_crecimiento",
      "afinidad_fondos_inversion",
      "canal_asesor_personal"
    ],
    "entonces": "Ofrecer asesoría financiera personalizada"
  },
  {
    "id": "R10",
    "si": [
      "rfm_monetary_bajo",
      "frequency_baja",
      "recency_muy_reciente",
      "arquetipo_profesional_joven_digital",
      "customer_journey_activacion",
      "afinidad_tarjeta_credito",
      "canal_aplicacion_movil"
    ],
    "entonces": "Campaña automática: Oferta de tarjeta crédito sin anualidad"
  },
  {
    "id": "R11",
    "si": [
      "rfm_monetary_alto",
      "frequency_alta",
      "arquetipo_cliente_patrimonial",
      "customer_journey_madurez",
      "afinidad_fondos_inversion",
      "canal_asesor_personal"
    ],
    "entonces": "Ofrecer productos de inversión avanzados"
  },
  {
    "id": "R12",
    "si": [
      "rfm_monetary_alto",
      "frequency_alta",
      "arquetipo_familia_en_expansion",
      "customer_journey_madurez",
      "afinidad_seguros",
      "canal_email"
    ],
    "entonces": "Ofrecer seguros familiares integrales"
  },
  {
    "id": "R13",
    "si": [
      "rfm_monetary_alto",
      "frequency_alta",
      "arquetipo_profesional_joven_digital",
      "customer_journey_madurez",
      "afinidad_fondos_inversion",
      "canal_aplicacion_movil"
    ],
    "entonces": "Plataforma premium de inversiones"
  },
  {
    "id": "R14",
    "si": [
      "rfm_monetary_alto",
      "frequency_alta",
      "arquetipo_emprendedor_pequeño_empresario",
      "customer_journey_madurez",
      "afinidad_credito_personal",
      "canal_asesor_personal"
    ],
    "entonces": "Líneas de crédito empresarial avanzadas"
  },
  {
    "id": "R15",
    "si": [
      "rfm_monetary_alto",
      "frequency_baja",
      "recency_antiguo",
      "arquetipo_cliente_patrimonial",
      "customer_journey_riesgo_de_abandono",
      "afinidad_fondos_inversion",
      "canal_asesor_personal"
    ],
    "entonces": "Contacto directo del gestor de cuenta"
  },
  {
    "id": "R16",
    "si": [
      "rfm_monetary_medio",
      "frequency_baja",
      "recency_muy_reciente",
      "arquetipo_familia_en_expansion",
      "customer_journey_crecimiento",
      "afinidad_credito_personal",
      "canal_email"
    ],
    "entonces": "Ofrecer crédito hipotecario o de consumo"
  },
  {
    "id": "R17",
    "si": [
      "rfm_monetary_bajo",
      "frequency_baja",
      "recency_antiguo",
      "arquetipo_cliente_transaccional",
      "customer_journey_madurez",
      "afinidad_cuenta_corriente",
      "canal_aplicacion_movil"
    ],
    "entonces": "Campaña automática: Cuenta corriente con beneficios básicos"
  },
  {
    "id": "R18",
    "si": [
      "rfm_monetary_bajo",
      "frequency_baja",
      "recency_inactivo",
      "arquetipo_cliente_transaccional",
      "customer_journey_reactivacion",
      "afinidad_cuenta_corriente",
      "canal_email"
    ],
    "entonces": "Campaña automática: Cuenta corriente con beneficios básicos"
  },
  {
    "id": "R19",
    "si": [
      "rfm_monetary_medio",
      "frequency_baja",
      "recency_antiguo",
      "arquetipo_emprendedor_pequeño_empresario",
      "customer_journey_crecimiento",
      "afinidad_credito_personal",
      "canal_telefono"
    ],
    "entonces": "Ofrecer líneas de crédito comercial"
  },
  {
    "id": "R20",
    "si": [
      "rfm_monetary_medio",
      "frequency_baja",
      "recency_muy_reciente",
      "arquetipo_profesional_joven_digital",
      "customer_journey_activacion",
      "afinidad_tarjeta_credito",
      "canal_aplicacion_movil"
    ],
    "entonces": "Campaña digital ofreciendo primera tarjeta con beneficios"
  },
  {
    "id": "R21",
    "si": [
      "rfm_monetary_alto",
      "frequency_baja",
      "recency_antiguo",
      "arquetipo_cliente_transaccional",
      "customer_journey_riesgo_de_abandono",
      "afinidad_cuenta_corriente",
      "canal_telefono"
    ],
    "entonces": "Campaña de retención personalizada"
  },
  {
    "id": "R22",
    "si": [
      "rfm_monetary_bajo",
      "frequency_baja",
      "recency_muy_reciente",
      "arquetipo_familia_en_expansion",
      "customer_journey_adquisicion",
      "afinidad_cuenta_corriente",
      "canal_sucursal"
    ],
    "entonces": "Campaña automática: Cuenta corriente con beneficios básicos"
  },
  {
    "id": "R23",
    "si": [
      "rfm_monetary_medio",
      "frequency_baja",
      "recency_muy_reciente",
      "arquetipo_profesional_joven_digital",
      "customer_journey_crecimiento",
      "afinidad_fondos_inversion",
      "canal_aplicacion_movil"
    ],
    "entonces": "Plataforma digital de inversiones"
  },
  {
    "id": "R24",
    "si": [
      "rfm_monetary_medio",
      "frequency_baja",
      "recency_antiguo",
      "arquetipo_emprendedor_pequeño_empresario",
      "customer_journey_riesgo_de_abandono",
      "afinidad_credito_personal",
      "canal_telefono"
    ],
    "entonces": "Contacto proactivo del asesor comercial"
  },
  {
    "id": "R25",
    "si": [
      "rfm_monetary_bajo",
      "frequency_baja",
      "recency_inactivo",
      "arquetipo_familia_en_expansion",
      "customer_journey_reactivacion",
      "afinidad_credito_personal",
      "canal_email"
    ],
    "entonces": "Campaña automática: Línea de crédito personal con tasa preferencial"
  },
  {
    "id": "R26",
    "si": [
      "rfm_monetary_alto",
      "frequency_baja",
      "recency_muy_reciente",
      "arquetipo_cliente_patrimonial",
      "customer_journey_adquisicion",
      "afinidad_fondos_inversion",
      "canal_asesor_personal"
    ],
    "entonces": "Bienvenida ejecutiva personalizada"
  },
  {
    "id": "R27",
    "si": [
      "rfm_monetary_bajo",
      "frequency_baja",
      "recency_muy_reciente",
      "arquetipo_emprendedor_pequeño_empresario",
      "customer_journey_adquisicion",
      "afinidad_cuenta_corriente",
      "canal_sucursal"
    ],
    "entonces": "Campaña automática: Cuenta corriente con beneficios básicos"
  },
  {
    "id": "R28",
    "si": [
      "rfm_monetary_bajo",
      "frequency_baja",
      "recency_muy_reciente",
      "arquetipo_cliente_transaccional",
      "customer_journey_adquisicion",
      "afinidad_cuenta_corriente",
      "canal_aplicacion_movil"
    ],
    "entonces": "Campaña automática: Cuenta corriente con beneficios básicos"
  },
  {
    "id": "R29",
    "si": [
      "rfm_monetary_bajo",
      "frequency_baja",
      "recency_antiguo",
      "arquetipo_profesional_joven_digital",
      "customer_journey_riesgo_de_abandono",
      "afinidad_tarjeta_credito",
      "canal_aplicacion_movil"
    ],
    "entonces": "Campaña automática: Oferta de tarjeta crédito sin anualidad"
  },
  {
    "id": "R30",
    "si": [
      "rfm_monetary_bajo",
      "frequency_baja",
      "recency_muy_reciente",
      "arquetipo_emprendedor_pequeño_empresario",
      "customer_journey_activacion",
      "afinidad_tarjeta_credito",
      "canal_telefono"
    ],
    "entonces": "Campaña automática: Oferta de tarjeta crédito sin anualidad"
  },
  {
    "id": "R31",
    "si": [
      "rfm_monetary_medio",
      "frequency_baja",
      "recency_antiguo",
      "arquetipo_familia_en_expansion",
      "customer_journey_riesgo_de_abandono",
      "afinidad_seguros",
      "canal_asesor_personal"
    ],
    "entonces": "Contacto de retención con asesor"
  },
  {
    "id": "R32",
    "si": [
      "rfm_monetary_bajo",
      "frequency_baja",
      "recency_inactivo",
      "arquetipo_cliente_transaccional",
      "customer_journey_reactivacion",
      "afinidad_cuenta_corriente",
      "canal_sms"
    ],
    "entonces": "Campaña automática: Cuenta corriente con beneficios básicos"
  },
  {
    "id": "R33",
    "si": [
      "rfm_monetary_medio",
      "frequency_baja",
      "recency_muy_reciente",
      "arquetipo_familia_en_expansion",
      "customer_journey_adquisicion",
      "afinidad_credito_personal",
      "canal_sucursal"
    ],
    "entonces": "Paquete de bienvenida familiar"
  },
  {
    "id": "R34",
    "si": [
      "rfm_monetary_medio",
      "frequency_alta",
      "recency_reciente",
      "arquetipo_cliente_patrimonial",
      "customer_journey_crecimiento",
      "afinidad_fondos_inversion",
      "canal_asesor_personal"
    ],
    "entonces": "Ofrecer productos de inversión intermedios"
  },
  {
    "id": "R35",
    "si": [
      "rfm_monetary_medio",
      "frequency_alta",
      "recency_reciente",
      "arquetipo_profesional_joven_digital",
      "customer_journey_crecimiento",
      "afinidad_fondos_inversion",
      "canal_aplicacion_movil"
    ],
    "entonces": "Acceso a portafolio de inversiones robo-advisor"
  },
  {
    "id": "R36",
    "si": [
      "rfm_monetary_medio",
      "frequency_media",
      "recency_reciente",
      "arquetipo_familia_en_expansion",
      "customer_journey_crecimiento",
      "afinidad_credito_personal",
      "canal_email"
    ],
    "entonces": "Ofertas de productos complementarios"
  },
  {
    "id": "R37",
    "si": [
      "rfm_monetary_alto",
      "frequency_media",
      "recency_reciente",
      "arquetipo_cliente_patrimonial",
      "customer_journey_madurez",
      "afinidad_fondos_inversion",
      "canal_asesor_personal"
    ],
    "entonces": "Revisión anual de portafolio"
  },
  {
    "id": "R38",
    "si": [
      "rfm_monetary_bajo",
      "frequency_media",
      "recency_reciente",
      "arquetipo_profesional_joven_digital",
      "customer_journey_activacion",
      "afinidad_tarjeta_credito",
      "canal_aplicacion_movil"
    ],
    "entonces": "Campaña automática: Oferta de tarjeta crédito sin anualidad"
  },
  {
    "id": "R39",
    "si": [
      "rfm_monetary_medio",
      "frequency_media",
      "recency_antiguo",
      "arquetipo_cliente_transaccional",
      "customer_journey_madurez",
      "afinidad_cuenta_corriente",
      "canal_sms"
    ],
    "entonces": "Comunicación segmentada con incentivos"
  },
  {
    "id": "R40",
    "si": [
      "rfm_monetary_alto",
      "frequency_media",
      "recency_reciente",
      "arquetipo_emprendedor_pequeño_empresario",
      "customer_journey_madurez",
      "afinidad_credito_personal",
      "canal_asesor_personal"
    ],
    "entonces": "Soluciones de tesorería avanzada"
  },
  {
    "id": "R41",
    "si": [
      "rfm_monetary_bajo",
      "frequency_media",
      "recency_antiguo",
      "arquetipo_familia_en_expansion",
      "customer_journey_madurez",
      "afinidad_credito_personal",
      "canal_email"
    ],
    "entonces": "Campaña automática: Línea de crédito personal con tasa preferencial"
  },
  {
    "id": "R42",
    "si": [
      "rfm_monetary_alto",
      "frequency_media",
      "recency_muy_reciente",
      "arquetipo_profesional_joven_digital",
      "customer_journey_adquisicion",
      "afinidad_tarjeta_credito",
      "canal_aplicacion_movil"
    ],
    "entonces": "Tarjeta premium con beneficios ejecutivos"
  },
  {
    "id": "R43",
    "si": [
      "rfm_monetary_medio",
      "frequency_media",
      "recency_reciente",
      "arquetipo_emprendedor_pequeño_empresario",
      "customer_journey_crecimiento",
      "afinidad_credito_personal",
      "canal_telefono"
    ],
    "entonces": "Línea de crédito flexible para negocio"
  },
  {
    "id": "R44",
    "si": [
      "rfm_monetary_bajo",
      "frequency_media",
      "recency_reciente",
      "arquetipo_cliente_transaccional",
      "customer_journey_crecimiento",
      "afinidad_cuenta_corriente",
      "canal_aplicacion_movil"
    ],
    "entonces": "Campaña automática: Cuenta corriente con beneficios básicos"
  },
  {
    "id": "R45",
    "si": [
      "rfm_monetary_medio",
      "frequency_media",
      "recency_antiguo",
      "arquetipo_cliente_patrimonial",
      "customer_journey_riesgo_de_abandono",
      "afinidad_fondos_inversion",
      "canal_asesor_personal"
    ],
    "entonces": "Contacto de reconexión con análisis de valor"
  }
]'''

print("Reglas y librerías cargadas exitosamente")

Reglas y librerías cargadas exitosamente


In [2]:
class SistemaCRM:
    """
    Sistema de recomendación CRM basado en reglas de conocimiento
    Utiliza RFM analysis, customer archetypes y journey stages para recomendar acciones

    ACTUALIZADO: Carga ahora las 45 REGLAS COMPLETAS del sistema
    - 8 Reglas con 5 condiciones (especificidad media - fallback)
    - 37 Reglas con 6-7 condiciones (especificidad alta - prioritarias)
    """

    def __init__(self):
        self.hechos = []
        self.reglas = []


        self.cargar_reglas_embebidas()

    def cargar_reglas_embebidas(self):
        """Carga las reglas embebidas en el script"""
        try:
            self.reglas = json.loads(REGLAS_JSON)
            print(f"[OK] Cargadas {len(self.reglas)} reglas desde REGLAS_JSON embebidas\n")
        except json.JSONDecodeError as e:
            print(f"[ERROR] Error: JSON embebido inválido: {e}")

    def agregar_hecho(self, hecho):
        """Agrega un hecho al conocimiento actual"""
        if hecho not in self.hechos:
            self.hechos.append(hecho)

    def agregar_hechos(self, hechos):
        """Agrega múltiples hechos"""
        for hecho in hechos:
            self.agregar_hecho(hecho)

    def limpiar_hechos(self):
        """Limpia todos los hechos"""
        self.hechos = []

    def inferir(self):
        """
        MOTOR DE INFERENCIA: Busca reglas que coincidan con los hechos actuales

        ALGORITMO DE RESOLUCIÓN DE CONFLICTOS:
        1. Evalúa todas las reglas contra hechos
        2. Retorna todas las reglas que aplican
        3. Ordena por especificidad (mayor # condiciones = mayor prioridad)
        """
        reglas_coincidentes = []

        for regla in self.reglas:
            # Verificar si todas las condiciones de la regla están en los hechos (lógica AND)
            if all(condicion in self.hechos for condicion in regla['si']):
                reglas_coincidentes.append(regla)

        # Ordenar por especificidad (mayor número de condiciones primero)
        reglas_coincidentes.sort(key=lambda r: len(r['si']), reverse=True)

        return reglas_coincidentes

    def mostrar_resultado(self, cliente_nombre="Cliente"):
        """Muestra los resultados de la inferencia con detalles de especificidad"""
        print(f"+" + "="*62 + "+")
        print(f"| ANALISIS CRM: {cliente_nombre:50} |")
        print(f"+" + "="*62 + "+\n")

        print(f"[HECHOS] HECHOS IDENTIFICADOS ({len(self.hechos)} condiciones):")
        for hecho in self.hechos:
            print(f"   - {hecho}")

        print(f"\n[BUSCANDO] BUSCANDO REGLAS APLICABLES...\n")

        reglas_aplicables = self.inferir()

        if reglas_aplicables:
            print(f"[ENCONTRADAS] Se encontraron {len(reglas_aplicables)} regla(s):\n")

            # Mostrar regla de máxima especificidad (debe aplicarse)
            regla_optima = reglas_aplicables[0]
            n_cond = len(regla_optima['si'])
            spec_label = "ALTA (6-7)" if n_cond >= 6 else "MEDIA (5)"

            print(f"{'*' * 65}")
            print(f"REGLA SELECCIONADA (Máxima Especificidad):")
            print(f"{'*' * 65}")
            print(f"Regla ID: {regla_optima['id']}")
            print(f"Especificidad: {spec_label} ({n_cond} condiciones)")
            print(f"Acción recomendada:\n  >> {regla_optima['entonces']}")
            print(f"{'*' * 65}\n")

            # Mostrar alternativas si existen
            if len(reglas_aplicables) > 1:
                print(f"Reglas alternativas ({len(reglas_aplicables) - 1} adicionales):\n")
                for i, regla in enumerate(reglas_aplicables[1:], 1):
                    n_cond = len(regla['si'])
                    spec_label = "ALTA (6-7)" if n_cond >= 6 else "MEDIA (5)"
                    print(f"{i}. {regla['id']} ({spec_label}): {regla['entonces'][:55]}...")
        else:
            print(f"[NO ENCONTRADAS] No se encontraron reglas que coincidan con estos hechos")

        print(f"\n{'=' * 65}\n")
        return reglas_aplicables


# Crear instancia del sistema CON LAS 45 REGLAS COMPLETAS
sistema = SistemaCRM()

print("="*65)
print("Sistema CRM inicializado con 45 reglas completas")
print("="*65)

[OK] Cargadas 45 reglas desde REGLAS_JSON embebidas

Sistema CRM inicializado con 45 reglas completas


## Representación de Clientes con Frames

En esta sección, implementamos un sistema de **Frames con Herencia** para representar clientes del banco.
Esto permite:
- Organizar información estructurada de clientes
- Heredar propiedades comunes (Persona → Cliente específico)
- Aplicar reglas CRM basadas en atributos del frame
- Demostrar conocimiento orientado a objetos

**Arquitectura de Frames:**
```
Persona (Frame Padre)
  ├─ nombre
  ├─ apellido
  ├─ edad
  └─ dirección

Cliente (hereda de Persona)
  ├─ numero_cuenta
  ├─ saldo
  ├─ rfm_monetary
  ├─ frequency
  ├─ recency
  ├─ customer_journey
  ├─ afinidad_producto
  └─ canal_contacto
```

In [3]:
from typing import Optional, Dict, Any

# ============================================================================
# CLASE FRAME: Representación de Conocimiento Estructurado con Herencia
# ============================================================================

class Frame:
    """
    Clase para representar un Frame (marco/esquema) con soporte para herencia.

    Los frames permiten organizar información sobre entidades del mundo real,
    con la capacidad de heredar atributos de frames padres.

    Atributos:
    - nombre: Identificador único del frame
    - slots: Diccionario de atributos (clave-valor)
    - padre: Frame padre del cual hereda valores
    - arquetipo: Tipo de cliente (para fines CRM)
    """

    def __init__(self, nombre: str, slots: Optional[Dict[str, Any]] = None,
                 padre: Optional['Frame'] = None, arquetipo: str = ""):
        self.nombre = nombre
        self.slots = slots if slots else {}
        self.padre = padre
        self.arquetipo = arquetipo

    def set_slot(self, atributo: str, valor: Any) -> None:
        """Asigna un valor a un slot del frame"""
        self.slots[atributo] = valor

    def get_slot(self, atributo: str) -> Any:
        """
        Obtiene el valor de un slot.
        Si no existe en el frame actual, busca en el frame padre (HERENCIA).
        """
        if atributo in self.slots:
            return self.slots[atributo]
        if self.padre:
            return self.padre.get_slot(atributo)
        return None

    def get_all_slots(self) -> Dict[str, Any]:
        """Obtiene todos los slots (propios + heredados)"""
        resultado = {}
        if self.padre:
            resultado.update(self.padre.get_all_slots())
        resultado.update(self.slots)
        return resultado

    def get_hechos_crm(self) -> list:
        """
        Convierte los slots del frame en hechos para el motor de inferencia CRM.
        ACTUALIZADO: Mapea correctamente los nombres de slots a condiciones de reglas.
        """
        hechos = []
        slots = self.get_all_slots()

        # Mapeo de slots a prefijos de hechos (para coincidencia con reglas)
        slot_to_fact_prefix = {
            'afinidad_producto': 'afinidad',      # afinidad_producto -> afinidad_*
            'canal_contacto': 'canal',            # canal_contacto -> canal_*
        }

        for atributo, valor in slots.items():
            if isinstance(valor, str) and valor not in ["No definido", "No definida"]:
                # Obtener el prefijo correcto para el hecho
                prefijo = slot_to_fact_prefix.get(atributo, atributo)

                # Formato: prefijo_valor (ej: rfm_monetary_alto, afinidad_fondos_inversion)
                hecho = f"{prefijo}_{valor}".lower().replace(" ", "_")
                hechos.append(hecho)
            elif isinstance(valor, bool) and valor:
                hechos.append(f"{atributo}_si")

        return hechos

    def __repr__(self) -> str:
        return f"Frame({self.nombre}) [{self.arquetipo}]"

    def __str__(self) -> str:
        slots_str = ", ".join([f"{k}={v}" for k, v in self.slots.items()])
        return f"{self.nombre}: {slots_str}"


print("Clase Frame actualizada con mapeo correcto de slots a hechos CRM")


Clase Frame actualizada con mapeo correcto de slots a hechos CRM


### Creación del Frame Padre: Persona

Definimos el frame base **Persona** con atributos comunes a todos los individuos.

In [4]:
# ============================================================================
# FRAME PADRE: Persona
# ============================================================================

persona = Frame(
    nombre="Persona",
    slots={
        "nombre": "No definido",
        "apellido": "No definido",
        "edad": 0,
        "direccion": "No definida"
    },
    padre=None
)

print("Frame Padre 'Persona' creado con atributos base")
print(f"  - nombre")
print(f"  - apellido")
print(f"  - edad")
print(f"  - dirección")


Frame Padre 'Persona' creado con atributos base
  - nombre
  - apellido
  - edad
  - dirección


### Creación de Frames de Clientes Específicos

Creamos diferentes clientes (frames hijos) que heredan de Persona.
Cada cliente tiene atributos RFM y propiedades CRM específicas.

In [5]:
# ============================================================================
# FRAMES DE CLIENTES: Heredan de Persona
# ============================================================================

# Cliente 1: Patrimonial - Alto Valor (ACTUALIZADO: includes frequency_media y customer_journey_crecimiento)
cliente_patrimonial = Frame(
    nombre="Cliente_Roberto",
    slots={
        "nombre": "Roberto",
        "apellido": "Martínez",
        "edad": 55,
        "direccion": "Escalante, San José",
        # Atributos CRM
        "numero_cuenta": "CTA-001",
        "arquetipo": "cliente_patrimonial",
        "rfm_monetary": "alto",
        "frequency": "media",
        "recency": "reciente",
        "customer_journey": "crecimiento",
        "afinidad_producto": "fondos_inversion",
        "canal_contacto": "asesor_personal",
        "saldo": 500000.00
    },
    padre=persona,
    arquetipo="cliente_patrimonial"
)

# Cliente 2: Profesional Joven Digital
cliente_digital = Frame(
    nombre="Cliente_Andrea",
    slots={
        "nombre": "Andrea",
        "apellido": "González",
        "edad": 28,
        "direccion": "Barrio Escalante, San José",
        # Atributos CRM
        "numero_cuenta": "CTA-002",
        "arquetipo": "profesional_joven_digital",
        "rfm_monetary": "bajo",
        "frequency": "baja",
        "recency": "muy_reciente",
        "customer_journey": "activacion",
        "afinidad_producto": "tarjeta_credito",
        "canal_contacto": "aplicacion_movil",
        "saldo": 15000.00
    },
    padre=persona,
    arquetipo="profesional_joven_digital"
)

# Cliente 3: Familia en Expansión (ACTUALIZADO: includes afinidad_seguros y canal_email)
cliente_familia = Frame(
    nombre="Cliente_familia_Lopez",
    slots={
        "nombre": "Carlos",
        "apellido": "López",
        "edad": 40,
        "direccion": "Moravia, San José",
        # Atributos CRM
        "numero_cuenta": "CTA-003",
        "arquetipo": "familia_en_expansion",
        "rfm_monetary": "medio",
        "frequency": "baja",
        "recency": "muy_reciente",
        "customer_journey": "adquisicion",
        "afinidad_producto": "seguros",
        "canal_contacto": "email",
        "saldo": 50000.00
    },
    padre=persona,
    arquetipo="familia_en_expansion"
)

# Cliente 4: Emprendedor
cliente_emprendedor = Frame(
    nombre="Cliente_Marina",
    slots={
        "nombre": "Marina",
        "apellido": "Rodríguez",
        "edad": 35,
        "direccion": "Heredia, Heredia",
        # Atributos CRM
        "numero_cuenta": "CTA-004",
        "arquetipo": "emprendedor_pequeño_empresario",
        "rfm_monetary": "medio",
        "frequency": "baja",
        "recency": "antiguo",
        "customer_journey": "riesgo_de_abandono",
        "afinidad_producto": "credito_personal",
        "canal_contacto": "telefono",
        "saldo": 75000.00
    },
    padre=persona,
    arquetipo="emprendedor_pequeño_empresario"
)

# Cliente 5: Transaccional Básico
cliente_transaccional = Frame(
    nombre="Cliente_Juan",
    slots={
        "nombre": "Juan",
        "apellido": "Pérez",
        "edad": 22,
        "direccion": "San Pedro, San José",
        # Atributos CRM
        "numero_cuenta": "CTA-005",
        "arquetipo": "cliente_transaccional",
        "rfm_monetary": "bajo",
        "frequency": "baja",
        "recency": "inactivo",
        "customer_journey": "reactivacion",
        "afinidad_producto": "cuenta_corriente",
        "canal_contacto": "sms",
        "saldo": 5000.00
    },
    padre=persona,
    arquetipo="cliente_transaccional"
)

clientes_frames = [
    cliente_patrimonial,
    cliente_digital,
    cliente_familia,
    cliente_emprendedor,
    cliente_transaccional
]


## Aplicación de Reglas CRM usando Frames

Demostramos cómo convertir los atributos de un Frame cliente en hechos para el motor de inferencia
y luego aplicar las reglas del sistema CRM.

In [6]:
# ============================================================================
# APLICAR REGLAS CRM A FRAMES DE CLIENTES
# ============================================================================

def aplicar_reglas_a_frame(cliente_frame: Frame, sistema_crm: SistemaCRM) -> Dict[str, Any]:
    """
    Aplica las reglas del sistema CRM a un cliente representado como Frame.

    Args:
        cliente_frame: Frame del cliente
        sistema_crm: Sistema CRM con las reglas cargadas

    Returns:
        Diccionario con información del cliente y reglas aplicables
    """

    # Convertir slots del frame a hechos CRM
    hechos = cliente_frame.get_hechos_crm()

    # También agregar informacion derivada del arquetipo
    hechos.append(f"arquetipo_{cliente_frame.get_slot('arquetipo')}")

    # Agregar hechos basados en el customer journey
    journey = cliente_frame.get_slot('customer_journey')
    if journey:
        hechos.append(f"customer_journey_{journey}")

    # Cargar hechos en el sistema CRM
    sistema_crm.limpiar_hechos()
    sistema_crm.agregar_hechos(hechos)

    # Ejecutar inferencia
    reglas_aplicables = sistema_crm.inferir()

    return {
        'cliente': cliente_frame.nombre,
        'slots': cliente_frame.get_all_slots(),
        'hechos': hechos,
        'reglas_aplicables': reglas_aplicables
    }


# Inicializar sistema CRM con las 45 reglas
sistema_crm = SistemaCRM()

print("\n" + "="*80)
print("APLICACIÓN DE REGLAS CRM A CLIENTES REPRESENTADOS COMO FRAMES")
print("="*80)

# Aplicar reglas a cada cliente
resultados = []
for cliente in clientes_frames:
    resultado = aplicar_reglas_a_frame(cliente, sistema_crm)
    resultados.append(resultado)

    # Mostrar resumen
    print(f"\n{'─'*80}")
    print(f"CLIENTE: {cliente.nombre}")
    print(f"{'─'*80}")
    print(f"Persona: {cliente.get_slot('nombre')} {cliente.get_slot('apellido')}, {cliente.get_slot('edad')} años")
    print(f"Dirección: {cliente.get_slot('direccion')} (HEREDADO de Persona)")
    print(f"Arqueipo: {cliente.arquetipo}")
    print(f"Cuenta: {cliente.get_slot('numero_cuenta')} | Saldo: ${cliente.get_slot('saldo'):,.2f}")
    print(f"\nHechos identificados: {len(resultado['hechos'])} hechos")
    print(f"Reglas aplicables: {len(resultado['reglas_aplicables'])} regla(s)")

    if resultado['reglas_aplicables']:
        regla_optima = resultado['reglas_aplicables'][0]
        print(f"\nREGLA SELECCIONADA (Máxima Especificidad):")
        print(f"   ID: {regla_optima['id']}")
        print(f"   Condiciones: {len(regla_optima['si'])}")
        print(f"   Acción: {regla_optima['entonces']}")

print("\n" + "="*80)
print("APLICACIÓN DE REGLAS COMPLETADA")
print("="*80)


[OK] Cargadas 45 reglas desde REGLAS_JSON embebidas


APLICACIÓN DE REGLAS CRM A CLIENTES REPRESENTADOS COMO FRAMES

────────────────────────────────────────────────────────────────────────────────
CLIENTE: Cliente_Roberto
────────────────────────────────────────────────────────────────────────────────
Persona: Roberto Martínez, 55 años
Dirección: Escalante, San José (HEREDADO de Persona)
Arqueipo: cliente_patrimonial
Cuenta: CTA-001 | Saldo: $500,000.00

Hechos identificados: 13 hechos
Reglas aplicables: 1 regla(s)

REGLA SELECCIONADA (Máxima Especificidad):
   ID: R1
   Condiciones: 5
   Acción: Ofrecer productos de inversión premium

────────────────────────────────────────────────────────────────────────────────
CLIENTE: Cliente_Andrea
────────────────────────────────────────────────────────────────────────────────
Persona: Andrea González, 28 años
Dirección: Barrio Escalante, San José (HEREDADO de Persona)
Arqueipo: profesional_joven_digital
Cuenta: CTA-002 | Saldo: $15,000.00

He

## Demostración mediante reglas de producción

In [7]:
# ========== EJEMPLO 1: Cliente Transaccional - Bajo Valor (Adquisición) ==========
print("\n" + "="*65)
print("EJEMPLO 1: Cliente Transaccional - Bajo Valor (Adquisición)")
print("="*65 + "\n")

sistema.limpiar_hechos()
sistema.agregar_hechos([
    "rfm_monetary_bajo",
    "frequency_baja",
    "recency_muy_reciente",
    "arquetipo_cliente_transaccional",
    "customer_journey_adquisicion",
    "afinidad_cuenta_corriente",
    "canal_aplicacion_movil"
])
sistema.mostrar_resultado("Cliente Transaccional #001")


EJEMPLO 1: Cliente Transaccional - Bajo Valor (Adquisición)

+==============================================================+
| ANALISIS CRM: Cliente Transaccional #001                         |
+==============================================================+

[HECHOS] HECHOS IDENTIFICADOS (7 condiciones):
   - rfm_monetary_bajo
   - frequency_baja
   - recency_muy_reciente
   - arquetipo_cliente_transaccional
   - customer_journey_adquisicion
   - afinidad_cuenta_corriente
   - canal_aplicacion_movil

[BUSCANDO] BUSCANDO REGLAS APLICABLES...

[ENCONTRADAS] Se encontraron 1 regla(s):

*****************************************************************
REGLA SELECCIONADA (Máxima Especificidad):
*****************************************************************
Regla ID: R28
Especificidad: ALTA (6-7) (7 condiciones)
Acción recomendada:
  >> Campaña automática: Cuenta corriente con beneficios básicos
*****************************************************************





[{'id': 'R28',
  'si': ['rfm_monetary_bajo',
   'frequency_baja',
   'recency_muy_reciente',
   'arquetipo_cliente_transaccional',
   'customer_journey_adquisicion',
   'afinidad_cuenta_corriente',
   'canal_aplicacion_movil'],
  'entonces': 'Campaña automática: Cuenta corriente con beneficios básicos'}]

In [8]:
# ========== EJEMPLO 2: Cliente Patrimonial - Alto Valor (Crecimiento) ==========
print("\n" + "="*65)
print("EJEMPLO 2: Cliente Patrimonial - Alto Valor (Crecimiento)")
print("="*65 + "\n")

sistema.limpiar_hechos()
sistema.agregar_hechos([
    "rfm_monetary_alto",
    "frequency_alta",
    "arquetipo_cliente_patrimonial",
    "customer_journey_crecimiento",
    "afinidad_fondos_inversion",
    "canal_asesor_personal"
])
sistema.mostrar_resultado("Cliente Patrimonial #002")


EJEMPLO 2: Cliente Patrimonial - Alto Valor (Crecimiento)

+==============================================================+
| ANALISIS CRM: Cliente Patrimonial #002                           |
+==============================================================+

[HECHOS] HECHOS IDENTIFICADOS (6 condiciones):
   - rfm_monetary_alto
   - frequency_alta
   - arquetipo_cliente_patrimonial
   - customer_journey_crecimiento
   - afinidad_fondos_inversion
   - canal_asesor_personal

[BUSCANDO] BUSCANDO REGLAS APLICABLES...

[ENCONTRADAS] Se encontraron 2 regla(s):

*****************************************************************
REGLA SELECCIONADA (Máxima Especificidad):
*****************************************************************
Regla ID: R9
Especificidad: ALTA (6-7) (6 condiciones)
Acción recomendada:
  >> Ofrecer asesoría financiera personalizada
*****************************************************************

Reglas alternativas (1 adicionales):

1. R1 (MEDIA (5)): Ofrecer productos

[{'id': 'R9',
  'si': ['rfm_monetary_alto',
   'frequency_alta',
   'arquetipo_cliente_patrimonial',
   'customer_journey_crecimiento',
   'afinidad_fondos_inversion',
   'canal_asesor_personal'],
  'entonces': 'Ofrecer asesoría financiera personalizada'},
 {'id': 'R1',
  'si': ['rfm_monetary_alto',
   'arquetipo_cliente_patrimonial',
   'customer_journey_crecimiento',
   'afinidad_fondos_inversion',
   'canal_asesor_personal'],
  'entonces': 'Ofrecer productos de inversión premium'}]

In [9]:
# ========== EJEMPLO 3: Cliente Digital - Riesgo de Abandono ==========
print("\n" + "="*65)
print("EJEMPLO 3: Cliente Digital - Riesgo de Abandono (Retención)")
print("="*65 + "\n")

sistema.limpiar_hechos()
sistema.agregar_hechos([
    "rfm_monetary_bajo",
    "frequency_baja",
    "recency_antiguo",
    "arquetipo_profesional_joven_digital",
    "customer_journey_riesgo_de_abandono",
    "afinidad_tarjeta_credito",
    "canal_aplicacion_movil"
])
sistema.mostrar_resultado("Cliente Digital #003 - EN RIESGO")


EJEMPLO 3: Cliente Digital - Riesgo de Abandono (Retención)

+==============================================================+
| ANALISIS CRM: Cliente Digital #003 - EN RIESGO                   |
+==============================================================+

[HECHOS] HECHOS IDENTIFICADOS (7 condiciones):
   - rfm_monetary_bajo
   - frequency_baja
   - recency_antiguo
   - arquetipo_profesional_joven_digital
   - customer_journey_riesgo_de_abandono
   - afinidad_tarjeta_credito
   - canal_aplicacion_movil

[BUSCANDO] BUSCANDO REGLAS APLICABLES...

[ENCONTRADAS] Se encontraron 1 regla(s):

*****************************************************************
REGLA SELECCIONADA (Máxima Especificidad):
*****************************************************************
Regla ID: R29
Especificidad: ALTA (6-7) (7 condiciones)
Acción recomendada:
  >> Campaña automática: Oferta de tarjeta crédito sin anualidad
*****************************************************************





[{'id': 'R29',
  'si': ['rfm_monetary_bajo',
   'frequency_baja',
   'recency_antiguo',
   'arquetipo_profesional_joven_digital',
   'customer_journey_riesgo_de_abandono',
   'afinidad_tarjeta_credito',
   'canal_aplicacion_movil'],
  'entonces': 'Campaña automática: Oferta de tarjeta crédito sin anualidad'}]

In [10]:
# ========== EJEMPLO 4: Emprendedor - Valor Medio (Crecimiento) ==========
print("\n" + "="*65)
print("EJEMPLO 4: Emprendedor - Valor Medio (Crecimiento)")
print("="*65 + "\n")

sistema.limpiar_hechos()
sistema.agregar_hechos([
    "rfm_monetary_medio",
    "frequency_baja",
    "recency_antiguo",
    "arquetipo_emprendedor_pequeño_empresario",
    "customer_journey_crecimiento",
    "afinidad_credito_personal",
    "canal_telefono"
])
sistema.mostrar_resultado("Emprendedor #004")



EJEMPLO 4: Emprendedor - Valor Medio (Crecimiento)

+==============================================================+
| ANALISIS CRM: Emprendedor #004                                   |
+==============================================================+

[HECHOS] HECHOS IDENTIFICADOS (7 condiciones):
   - rfm_monetary_medio
   - frequency_baja
   - recency_antiguo
   - arquetipo_emprendedor_pequeño_empresario
   - customer_journey_crecimiento
   - afinidad_credito_personal
   - canal_telefono

[BUSCANDO] BUSCANDO REGLAS APLICABLES...

[ENCONTRADAS] Se encontraron 2 regla(s):

*****************************************************************
REGLA SELECCIONADA (Máxima Especificidad):
*****************************************************************
Regla ID: R19
Especificidad: ALTA (6-7) (7 condiciones)
Acción recomendada:
  >> Ofrecer líneas de crédito comercial
*****************************************************************

Reglas alternativas (1 adicionales):

1. R6 (MEDIA (5)): Con

[{'id': 'R19',
  'si': ['rfm_monetary_medio',
   'frequency_baja',
   'recency_antiguo',
   'arquetipo_emprendedor_pequeño_empresario',
   'customer_journey_crecimiento',
   'afinidad_credito_personal',
   'canal_telefono'],
  'entonces': 'Ofrecer líneas de crédito comercial'},
 {'id': 'R6',
  'si': ['rfm_monetary_medio',
   'arquetipo_emprendedor_pequeño_empresario',
   'customer_journey_crecimiento',
   'afinidad_credito_personal',
   'canal_telefono'],
  'entonces': 'Consultoría de crédito comercial personalizada'}]